In [3]:
import pandas as pd
import pyodbc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from category_encoders import TargetEncoder
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, recall_score, precision_score

### Connect Python to SQL Server

In [4]:
import pyodbc
import pandas as pd

conn = pyodbc.connect(
    "DRIVER={SQL Server};"
    "SERVER=localhost\\SQLEXPRESS;"
    "DATABASE=finance;"
    "Trusted_Connection=yes;"
)


### After sampling in sql -> load only sampled data into python

In [5]:
query ="""
SELECT *
FROM vw_ml_transactions
WHERE fraud_label = 1

UNION ALL

SELECT *
FROM (
    SELECT TOP 500000 *
    FROM vw_ml_transactions
    WHERE fraud_label = 0
    ORDER BY NEWID()
) AS sampled_nonfraud;
"""

df = pd.read_sql_query(query,conn)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_14840\1473978831.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query,conn)


In [6]:
df.head()

,amount,use_chip,errors,fraud_label,card_id,transaction_date,current_age,retirement_age,gender,per_capita_income,...,card_type,num_cards_issued,credit_limit,acct_open_date,mcc_description,day,month,quarter,weekend_flag,year
0,53.01,Online Transaction,False,True,117,2010-10-21 07:20:00,20,71,Male,14824.0,...,Debit,1,10982.0,Jan-06,Steelworks,21,10,Q4,No,2010
1,371.24,Online Transaction,False,True,122,2016-03-27 14:53:00,65,66,Female,19237.0,...,Credit,1,14600.0,Jan-07,Steelworks,27,3,Q1,Yes,2016
2,128.06,Online Transaction,False,True,201,2016-01-21 13:51:00,61,65,Male,17567.0,...,Debit,1,6326.0,Jan-09,Steelworks,21,1,Q1,No,2016
3,380.81,Online Transaction,False,True,255,2010-05-09 12:52:00,18,57,Female,21214.0,...,Debit,1,66.0,Jan-10,Steelworks,9,5,Q2,Yes,2010
4,7.96,Online Transaction,False,True,1020,2014-02-23 12:57:00,29,66,Female,26545.0,...,Debit,2,14216.0,Feb-03,Steelworks,23,2,Q1,Yes,2014


In [7]:
df.shape

(502347, 26)

### Save locally

In [8]:
df.to_parquet("fraud_ml_dataset.parquet", index=False)

In [10]:
# RETRIEVE FROM PARQUET
df = pd.read_parquet("fraud_ml_dataset.parquet")

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 502347 entries, 0 to 502346
Data columns (total 26 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   amount             502347 non-null  float64       
 1   use_chip           502347 non-null  str           
 2   errors             502347 non-null  bool          
 3   fraud_label        502347 non-null  bool          
 4   card_id            502347 non-null  int64         
 5   transaction_date   502347 non-null  datetime64[us]
 6   current_age        502347 non-null  int64         
 7   retirement_age     502347 non-null  int64         
 8   gender             502347 non-null  str           
 9   per_capita_income  502347 non-null  float64       
 10  yearly_income      502347 non-null  float64       
 11  total_debt         502347 non-null  float64       
 12  credit_score       502347 non-null  int64         
 13  latitude           502347 non-null  float64       
 14 

In [12]:
df.describe()

,amount,card_id,transaction_date,current_age,retirement_age,per_capita_income,yearly_income,total_debt,credit_score,latitude,longitude,num_cards_issued,credit_limit,day,month,year
count,502347.000000,502347.000000,502347,502347.000000,502347.000000,502347.000000,502347.000000,502347.000000,502347.000000,502347.000000,502347.000000,502347.000000,502347.000000,502347.000000,502347.000000,502347.000000
mean,43.293048,708.858862,2015-01-04 20:52:36.998488,45.133376,66.105093,22701.184404,45232.507731,66094.585522,707.736121,36.974275,-92.533667,1.555924,14995.368072,15.711592,6.417946,2014.519085
min,-500.000000,0.000000,2010-01-01 00:02:00,18.000000,50.000000,0.000000,2.000000,0.000000,489.000000,21.340000,-158.010000,1.000000,0.000000,1.000000,1.000000,2010.000000
25%,9.000000,199.000000,2012-07-24 09:07:00,31.000000,65.000000,16413.000000,31739.000000,26097.000000,676.000000,33.760000,-103.230000,1.000000,7589.000000,8.000000,3.000000,2012.000000
50%,29.400000,1001.000000,2015-01-30 11:40:00,44.000000,66.000000,19652.000000,39214.000000,58509.000000,711.000000,37.630000,-87.340000,2.000000,13356.000000,16.000000,6.000000,2015.000000
75%,65.320000,1168.000000,2017-06-21 07:46:30,58.000000,68.000000,25584.000000,51581.000000,93592.000000,751.000000,40.770000,-79.990000,2.000000,20195.000000,23.000000,9.000000,2017.000000
max,4175.850000,1391.000000,2019-10-31 23:33:00,99.000000,75.000000,150583.000000,307018.000000,516263.000000,850.000000,61.140000,-70.080000,3.000000,98100.000000,31.000000,12.000000,2019.000000
std,82.777873,499.880284,NaN,17.812221,3.655134,12794.460951,26732.294296,56734.478381,69.354191,5.110156,17.067794,0.518611,11534.209635,8.792157,3.402654,2.835852


In [13]:
df.describe(include='object')

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_14840\87514550.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include='object')


,use_chip,gender,card_brand,card_type,acct_open_date,mcc_description,quarter,weekend_flag
count,502347,502347,502347,502347,502347,502347,502347,502347
unique,3,2,4,2,49,108,4,2
top,Swipe Transaction,Female,Mastercard,Debit,Feb-10,"Grocery Stores, Supermarkets",Q3,No
freq,266478,280870,258160,353066,55025,60781,129013,358352


In [14]:
df.acct_open_date.dtype # df['acct_open_date'].unique() - 'Jan-06', 'Jan-07'

<StringDtype(na_value=nan)>

In [15]:
df['acct_open_date'] = pd.to_datetime(
    df['acct_open_date'],
    format='%b-%y',
    errors='coerce'
)

In [16]:
df['acct_open_date'].dtypes

dtype('<M8[us]')

### Feature Engineering / Feature Construction

In [17]:
# Average spending
df['avg_spend'] = df.groupby('card_id')['amount'].transform('mean')

In [18]:
# Failed transactions ----
df['failed_transaction_count'] = df.groupby('card_id')['errors'].transform('sum')

In [19]:
df['hour'] = df['transaction_date'].dt.hour

In [20]:
# Night transaction risk feature
# Flag transactions between 10 PM (22) and 6 AM (6)
df['night_transaction_flag'] = df['hour'].apply(lambda h: 1 if (h>=22 or h<6) else 0)

# NIGHT WEEKEND FLAG risk feature
df['night_weekend_flag'] = ((df['night_transaction_flag'] == 1) & (df['weekend_flag'] ==1)).astype(int)

In [21]:
# RISKY MERCHANT CATEGORY (MCC (Merchant Category Code))
risky_mcc = [ "Betting (including Lottery Tickets, Casinos)",
              "Drinking Places (Alcoholic Beverages)",
                "Package Stores, Beer, Wine, Liquor",
                "Precious Stones and Metals",
                "Antique Shops",
                "Lodging - Hotels, Motels, Resorts",
                "Music Stores - Musical Instruments"
]
df['merchant_risk_flag'] = df['mcc_description'].isin(risky_mcc).astype(int)
df['merchant_risk_frequency'] = df.groupby('card_id')['merchant_risk_flag'].transform('sum')

acct_open_date is different: it tells you how long the account has been active. That’s a strong risk/fraud feature (e.g., new accounts are often riskier).You don’t need the raw date itself, but you can transform it into something useful like:

Account age in days/months/years = transaction_date - acct_open_date

Account age bucket (new, medium, old)

Here’s a simple way to turn acct_open_date into an account age feature using your existing transaction_date column: New accounts (low age) are often riskier for fraud or default.

Older accounts tend to be more stable.

In [22]:
# Calculate account age in days at the time of transaction
df['account_age_days'] = (df['transaction_date'] - df['acct_open_date']).dt.days

# convert to months or years
df['account_age_months'] = df['account_age_days'] // 30
df['account_age_years'] = df['account_age_days'] // 365


In [23]:
df['account_age_days'].isna().sum()

np.int64(0)

In [24]:
df.shape

(502347, 36)

In [25]:
df = df[['amount', 'use_chip', 'fraud_label', 'current_age', 'retirement_age', 'gender',
       'per_capita_income', 'yearly_income', 'total_debt', 'credit_score',
       'latitude', 'longitude', 'card_brand', 'card_type', 'num_cards_issued',
       'credit_limit', 'mcc_description', 'day', 'month',
       'quarter', 'weekend_flag', 'year', 'avg_spend',
       'failed_transaction_count', 'night_transaction_flag',
       'night_weekend_flag', 'merchant_risk_flag', 'merchant_risk_frequency',
       'account_age_days', 'account_age_months', 'account_age_years']]

In [26]:
df.shape

(502347, 31)

### Train test split

In [27]:
fraud_percent = df['fraud_label'].value_counts(normalize=True)*100
fraud_percent

fraud_label
False    99.532793
True      0.467207
Name: proportion, dtype: float64

In [28]:
X = df.drop(columns='fraud_label')
y = df['fraud_label']
X_train,X_test,y_train,y_test = train_test_split(X,y,stratify=y, test_size=0.2, random_state=67)

OneHotEncoder for categorical features like use_chip, gender, card_brand, card_type.

Use TargetEncoder for high-cardinality categorical features like mcc_description.

In [29]:
encoder = TargetEncoder(cols=['mcc_description'])
X_train['mcc_description'] = encoder.fit_transform(X_train['mcc_description'], y_train)
X_test['mcc_description'] = encoder.transform(X_test['mcc_description'])

In [30]:
# Drop the first category of each feature to avoid multicollinearity and to deal with sparse feature use sparse_output
one_encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False,
    drop='first'   # <-- correct usage
)

# Fit
one_encoder.fit(X_train[['use_chip','gender','card_brand','card_type','quarter','weekend_flag']])

# Transform
X_train_onehot = pd.DataFrame(
    one_encoder.transform(X_train[['use_chip','gender','card_brand','card_type','quarter','weekend_flag']]),
    columns=one_encoder.get_feature_names_out()
)

X_test_onehot = pd.DataFrame(
    one_encoder.transform(X_test[['use_chip','gender','card_brand','card_type','quarter','weekend_flag']]),
    columns=one_encoder.get_feature_names_out()
)


merge original X_train (with numeric + date features) and the one-hot encoded categorical features (X_train_onehot), but avoid duplicating the original categorical columns.

In [31]:
# Drop the original categorical columns from X_train
X_train_num = X_train.drop(['use_chip','gender','card_brand','card_type','quarter','weekend_flag'], axis=1)

# Concatenate numeric/date features with one-hot encoded features
X_train_final = pd.concat([X_train_num.reset_index(drop=True), 
                           X_train_onehot.reset_index(drop=True)], axis=1)

# Do the same for test set
X_test_num = X_test.drop(['use_chip','gender','card_brand','card_type','quarter','weekend_flag'], axis=1)
X_test_final = pd.concat([X_test_num.reset_index(drop=True), 
                          X_test_onehot.reset_index(drop=True)], axis=1)


In [32]:
X_test_final.shape

(100470, 35)

In [33]:
# Select columns to scale
numeric_cols = [
    'amount', 'current_age', 'retirement_age', 'per_capita_income',
    'yearly_income', 'total_debt', 'credit_score', 'latitude', 'longitude',
    'num_cards_issued', 'credit_limit', 'day', 'month', 'year',
    'avg_spend', 'failed_transaction_count', 'merchant_risk_frequency',
    'account_age_days', 'account_age_months', 'account_age_years'
]

# Initialize scaler
scaler = StandardScaler()

# Fit on train and transform
X_train_scaled = X_train_final.copy()
X_test_scaled = X_test_final.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train_final[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test_final[numeric_cols])


In [34]:
X_train_scaled.shape

(401877, 35)

In [35]:
X_train_scaled.head(2)

,amount,current_age,retirement_age,per_capita_income,yearly_income,total_debt,credit_score,latitude,longitude,num_cards_issued,...,use_chip_Swipe Transaction,gender_Male,card_brand_Discover,card_brand_Mastercard,card_brand_Visa,card_type_Debit,quarter_Q2,quarter_Q3,quarter_Q4,weekend_flag_Yes
0,-0.253106,-1.074449,0.792718,-0.303255,-0.256661,-0.038091,1.877522,1.389351,-0.628030,-1.07167,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,0.024463,-0.288450,-0.577559,1.125255,1.137590,0.969427,0.609389,0.858464,1.132559,-1.07167,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0


In [36]:
lg = LogisticRegression(max_iter=5000)
lg.fit(X_train_scaled,y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [37]:
y_pred = lg.predict(X_test_scaled)
auc = roc_auc_score(y_test, y_pred)
print("Test AUC:", auc)

Test AUC: 0.5190997663584133


In [38]:
# -----------------------------
#  Create LightGBM datasets
# -----------------------------
train_data = lgb.Dataset(X_train_scaled, label=y_train)
test_data = lgb.Dataset(X_test_scaled, label=y_test)

# -----------------------------
# Define parameters
# -----------------------------
params = {
    'objective': 'binary',
    'boosting_type': 'gbdt',
    'metric': 'auc',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1
}

# -----------------------------
# Train model
# -----------------------------
model = lgb.train(
    params,
    train_data,
    valid_sets=[train_data, test_data],
    num_boost_round=500,
)

# -----------------------------
# Evaluate
# -----------------------------
y_pred = model.predict(X_test_scaled)
auc = roc_auc_score(y_test, y_pred)
print("Test AUC:", auc)


Test AUC: 0.9900217842171259


In [39]:
import numpy as np
from sklearn.metrics import recall_score, precision_score

# 1. Convert probabilities to binary labels (0 or 1)
y_pred_binary = (y_pred >= 0.5).astype(int)

# 2. Calculate metrics using the binary labels
recall = recall_score(y_test, y_pred_binary)
precision = precision_score(y_test, y_pred_binary)

print(f"Recall: {recall:.4f}")
print(f"Precision: {precision:.4f}")

Recall: 0.5757
Precision: 0.8852


In [43]:
import optuna
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, recall_score, precision_score

def objective(trial):
    params = {
        'objective': 'binary',
        'boosting_type': 'gbdt',
        'metric': 'auc',
        'verbose': -1,
        'num_leaves': trial.suggest_int('num_leaves', 20, 200),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000)
    }
    
    model = LGBMClassifier(**params)
    model.fit(X_train_scaled, y_train)
    
    # Predict probabilities
    y_pred = model.predict_proba(X_test_scaled)[:, 1]
    
    # Convert to binary labels with threshold 0.5
    y_pred_binary = (y_pred >= 0.5).astype(int)
    
    # Calculate metrics
    auc = roc_auc_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred_binary)
    precision = precision_score(y_test, y_pred_binary)
    
    # Print metrics for each trial
    print(f"AUC: {auc:.4f}, Recall: {recall:.4f}, Precision: {precision:.4f}")
    
    # Optuna will optimize based on AUC (you can change to recall/precision if desired)
    return auc

# Run optimization
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

print("Best parameters:", study.best_params)
print("Best AUC:", study.best_value)


[I 2026-05-11 15:34:30,210] A new study created in memory with name: no-name-25278876-f3e7-481b-81b3-8979c0a5292c
[I 2026-05-11 15:35:33,129] Trial 0 finished with value: 0.9954732009183106 and parameters: {'num_leaves': 124, 'learning_rate': 0.035695889227900866, 'feature_fraction': 0.7459957849850918, 'bagging_fraction': 0.7775223027156001, 'bagging_freq': 7, 'n_estimators': 806}. Best is trial 0 with value: 0.9954732009183106.


AUC: 0.9955, Recall: 0.6333, Precision: 0.9369


[I 2026-05-11 15:35:44,413] Trial 1 finished with value: 0.9606112680877456 and parameters: {'num_leaves': 53, 'learning_rate': 0.08995808871881873, 'feature_fraction': 0.8550004675762419, 'bagging_fraction': 0.9546556521325461, 'bagging_freq': 2, 'n_estimators': 162}. Best is trial 0 with value: 0.9954732009183106.


AUC: 0.9606, Recall: 0.5011, Precision: 0.6334


[I 2026-05-11 15:35:53,618] Trial 2 finished with value: 0.9795562278918789 and parameters: {'num_leaves': 33, 'learning_rate': 0.03705779150526631, 'feature_fraction': 0.8834832212354251, 'bagging_fraction': 0.6280913498603183, 'bagging_freq': 7, 'n_estimators': 197}. Best is trial 0 with value: 0.9954732009183106.


...

AUC: 0.9944, Recall: 0.6034, Precision: 0.9465


[I 2026-05-11 15:53:35,163] Trial 18 finished with value: 0.9945012276956121 and parameters: {'num_leaves': 200, 'learning_rate': 0.04982325513188245, 'feature_fraction': 0.7205381606660113, 'bagging_fraction': 0.660375254605139, 'bagging_freq': 10, 'n_estimators': 410}. Best is trial 0 with value: 0.9954732009183106.


AUC: 0.9945, Recall: 0.5991, Precision: 0.9305


[I 2026-05-11 15:55:15,820] Trial 19 finished with value: 0.9942765391109415 and parameters: {'num_leaves': 133, 'learning_rate': 0.023943131139528014, 'feature_fraction': 0.9060388800738525, 'bagging_fraction': 0.8179012039587792, 'bagging_freq': 7, 'n_estimators': 714}. Best is trial 0 with value: 0.9954732009183106.


AUC: 0.9943, Recall: 0.5970, Precision: 0.9556
Best parameters: {'num_leaves': 124, 'learning_rate': 0.035695889227900866, 'feature_fraction': 0.7459957849850918, 'bagging_fraction': 0.7775223027156001, 'bagging_freq': 7, 'n_estimators': 806}
Best AUC: 0.9954732009183106


In [44]:
# Train final model with best params
best_model = LGBMClassifier(**study.best_params)
best_model.fit(X_train_scaled, y_train)


,boosting_type,'gbdt'
,num_leaves,124
,max_depth,-1
,learning_rate,0.035695889227900866
,n_estimators,806
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [45]:
# Final evaluation
y_pred = best_model.predict_proba(X_test_scaled)[:, 1]
y_pred_binary = (y_pred >= 0.5).astype(int)

final_recall = recall_score(y_test, y_pred_binary)
final_precision = precision_score(y_test, y_pred_binary)

print(f"Final Recall: {final_recall:.4f}")
print(f"Final Precision: {final_precision:.4f}")


Final Recall: 0.6333
Final Precision: 0.9369


LightGBM's model.predict() function returns probabilities (continuous values between 0 and 1) by default when using the binary objective.

While roc_auc_score is happy to work with these probabilities, precision_score and recall_score are classification metrics—they expect hard labels (0 or 1, True or False) to build a confusion matrix. To calculate Recall or Precision, convert the continuous probabilities in y_pred into binary predictions using a threshold (typically 0.5). 
